## Import libraries

In [10]:
from pathlib import Path

from torchvision.datasets import ImageFolder
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from transformations import train_transform, val_transform

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)

cuda


## CNN


In [3]:
from torch import nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Flatten(),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.LazyLinear(53)
        )


    def forward(self, X):
        return self.network(X)

In [4]:
model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [8]:
path = Path("../VolleyballDataset")

if not path.exists():
   print("Dataset not found")
else:
    print("Volleyball Dataset Found")

BATCH = 32

torch.manual_seed(12345)

train_set = ImageFolder(path / "train", train_transform)
val_set = ImageFolder(path / "valid", val_transform)

# Data Loaders
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH)

Volleyball Dataset Found


In [14]:

def train(model, train_loader, val_loader, loss_fn, optimizer, epochs=10):
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)
        history["train_loss"].append(avg_train_loss)

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_accuracy = 100 * correct / total
        history['val_loss'].append(val_accuracy)
        
        print(f"Epoch {epoch+1} Summary: Train Loss: {avg_train_loss:.4f} | Val Acc: {val_accuracy:.2f}%")
        
    return history

In [15]:

history = train(model, train_loader, val_loader, loss_fn, optimizer, epochs=10)

Epoch 1/10: 100%|██████████| 400/400 [01:19<00:00,  5.02batch/s, loss=0.626]


Epoch 1 Summary: Train Loss: 0.6973 | Val Acc: 68.83%


Epoch 2/10: 100%|██████████| 400/400 [01:07<00:00,  5.94batch/s, loss=0.651]


Epoch 2 Summary: Train Loss: 0.5782 | Val Acc: 73.96%


Epoch 3/10: 100%|██████████| 400/400 [01:02<00:00,  6.44batch/s, loss=0.41] 


...

Epoch 8 Summary: Train Loss: 0.3431 | Val Acc: 77.26%


Epoch 9/10: 100%|██████████| 400/400 [01:01<00:00,  6.48batch/s, loss=0.194] 


Epoch 9 Summary: Train Loss: 0.3277 | Val Acc: 81.62%


Epoch 10/10: 100%|██████████| 400/400 [01:02<00:00,  6.43batch/s, loss=0.346] 


Epoch 10 Summary: Train Loss: 0.3115 | Val Acc: 79.59%
